# Fig. 2 u64 d2p15 EMA Check

Focused check for the long-trained `nf_fig2_u64_d2p15_noaug_200k` model at the largest dataset size.

This notebook now uses a denser post-hoc EMA grid so we can see whether the improvement has a local minimum rather than just comparing `raw`, `0.02`, and `0.10`.

Default labels:

`raw, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.008, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.075, 0.10`

Existing files are scored immediately; missing files are listed in the audit table and ignored by the metric plots until you sample them. Lower `hist_l1` and lower `pk_log10_mae` are better.


In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', Path.cwd())).resolve()
if not (PROJECT_DIR / 'scripts').exists():
    PROJECT_DIR = Path('/home/jiamingp/diffusion_models_repo')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.metrics import batch_power_spectra, field_histogram
from scripts.check_nf_generalize_fig2_small_ema import (
    DEFAULT_LABELS,
    evenly_limit,
    ema_value_for_label,
    load_manifest_row,
    load_npz_samples,
    load_real_lightweight,
    sample_path,
)

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180})

RUN_NAME = 'nf_fig2_u64_d2p15_noaug_200k'
SAMPLER = 'train_full'
SEED = 123
LABELS = list(DEFAULT_LABELS)
MAX_GENERATED = 128
MAX_REAL_CUBES = 8
MAX_REAL_HIST = 512
MAX_REAL_PK = 256
PK_NBINS = 25

MANIFEST_PATH = PROJECT_DIR / 'local/nf_generalize_fig2/manifest.json'
SAMPLE_ROOT = PROJECT_DIR / 'results/nf_generalize_fig2/samples'
OUT_DIR = PROJECT_DIR / 'results/nf_generalize_fig2/quickcheck'
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_CSV = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_audit.csv'
METRICS_CSV = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_metrics.csv'
SWEEP_PNG = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_sweep.png'

row = load_manifest_row(MANIFEST_PATH, RUN_NAME)
CONFIG_PATH = PROJECT_DIR / row['config']

print('project:', PROJECT_DIR)
print('run:', RUN_NAME)
print('config:', CONFIG_PATH)
print('sample root:', SAMPLE_ROOT)
print('output dir:', OUT_DIR)
print('ema labels:', ', '.join(LABELS))


## Sample Audit

This checks exactly which raw and EMA sample files are present. The raw file may have 512 samples from earlier; EMA files are expected to have 128 samples from the focused post-hoc EMA run. All metrics use the same `MAX_GENERATED=128` cap so the comparison is fair.

Missing rows are not an error for the notebook; they are the EMA points you still need to reconstruct with the Slurm command at the bottom.


In [ ]:
def npz_n(path: Path) -> int:
    if not path.exists():
        return 0
    with np.load(path, allow_pickle=True) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        return int(data[key].shape[0])

sample_rows = []
for label in LABELS:
    path = sample_path(SAMPLE_ROOT, RUN_NAME, SEED, label, SAMPLER)
    n = npz_n(path)
    sample_rows.append({
        'label': label,
        'ema_sigma_rel': ema_value_for_label(label),
        'exists': path.exists(),
        'n_available': n,
        'n_used_for_metrics': min(n, MAX_GENERATED),
        'size_mb': path.stat().st_size / 1024**2 if path.exists() else np.nan,
        'path': str(path),
    })

audit_df = pd.DataFrame(sample_rows)
display(audit_df)

missing = audit_df.query('n_available == 0')
present = audit_df.query('n_available > 0')
print(f'present EMA/sample files: {len(present)}/{len(audit_df)}')
if len(missing):
    print('Missing EMA samples; run the Slurm sampler before interpreting the full curve:')
    display(missing[['label', 'ema_sigma_rel', 'path']])


## Run / Refresh Scoring

The scoring script writes the audit CSV, metrics CSV, and the compact EMA sweep plot used below. It compares generated samples against the real training reference loaded from the same Fig. 2 config.


In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_DIR / 'scripts/check_nf_generalize_fig2_small_ema.py'),
    '--project-dir', str(PROJECT_DIR),
    '--run-name', RUN_NAME,
    '--sampler', SAMPLER,
    '--labels', ','.join(LABELS),
    '--seed', str(SEED),
    '--max-generated', str(MAX_GENERATED),
    '--max-real-cubes', str(MAX_REAL_CUBES),
    '--max-real-hist', str(MAX_REAL_HIST),
    '--max-real-pk', str(MAX_REAL_PK),
    '--pk-nbins', str(PK_NBINS),
]
print(' '.join(map(str, cmd)))
result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('stderr:')
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'scoring failed with return code {result.returncode}')


## Metric Summary

Interpretation:

- `hist_l1`: L1 distance between generated and real one-point PDFs. Lower is better.
- `pk_log10_mae`: mean absolute log10 error of generated mean P(k) vs real mean P(k). Lower is better.
- `std_ratio`: generated pixel std divided by real pixel std. Near 1 is better.
- `pk_ratio_low_k`, `pk_ratio_mid_k`, `pk_ratio_high_k`: broad-band P(k) ratios. Near 1 is better.

If raw remains best, EMA is not helping this long-trained checkpoint. If one EMA improves P(k) but worsens the one-point PDF, it is a tradeoff, not a clear win.


In [ ]:
metrics = pd.read_csv(METRICS_CSV)
metrics = metrics.sort_values('ema_value', na_position='first').reset_index(drop=True)
cols = [
    'ema_label', 'ema_value', 'n_available', 'n_used',
    'hist_l1', 'pk_log10_mae', 'std_ratio',
    'pk_ratio_low_k', 'pk_ratio_mid_k', 'pk_ratio_high_k', 'sample_path',
]
metrics_view = metrics[cols].copy()
display(metrics_view.round(5))

best_hist = metrics.sort_values(['hist_l1', 'pk_log10_mae']).iloc[0]
best_pk = metrics.sort_values(['pk_log10_mae', 'hist_l1']).iloc[0]
print('best one-point:', best_hist['ema_label'], 'hist_l1=', round(float(best_hist['hist_l1']), 5), 'pk_log10_mae=', round(float(best_hist['pk_log10_mae']), 5))
print('best P(k):     ', best_pk['ema_label'], 'pk_log10_mae=', round(float(best_pk['pk_log10_mae']), 5), 'hist_l1=', round(float(best_pk['hist_l1']), 5))

if SWEEP_PNG.exists():
    display(Image(filename=str(SWEEP_PNG)))
else:
    print('saved sweep PNG not found yet:', SWEEP_PNG)


## Raw vs EMA Metric Curves

These plots are generated directly from the metrics table, so they should display even if the saved PNG preview is unavailable. The minimum marker identifies the best sampled EMA value for each metric.


In [ ]:
raw = metrics[metrics['ema_label'] == 'raw']
ema = metrics[metrics['ema_label'] != 'raw'].sort_values('ema_value')
raw_x = float(ema['ema_value'].min()) / 2 if len(ema) else 0.01

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
plot_specs = [
    ('hist_l1', 'one-point L1', 'lower is better'),
    ('pk_log10_mae', 'P(k) log10 MAE', 'lower is better'),
    ('std_ratio', 'generated / real std', 'near 1 is better'),
]
for ax, (col, ylabel, subtitle) in zip(axes.flat[:3], plot_specs):
    if len(ema):
        ax.plot(ema['ema_value'], ema[col], marker='o', color='tab:blue', label='EMA')
        if col in {'hist_l1', 'pk_log10_mae'}:
            best = ema.sort_values(col).iloc[0]
            ax.scatter([best['ema_value']], [best[col]], s=90, marker='D', color='tab:red', label=f"best {best['ema_label']}")
    if len(raw):
        ax.scatter([raw_x], [float(raw.iloc[0][col])], marker='*', s=180, color='black', label='raw')
    if col == 'std_ratio':
        ax.axhline(1.0, color='0.4', lw=1, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel('EMA sigma_rel')
    ax.set_ylabel(ylabel)
    ax.set_title(subtitle)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

ax = axes.flat[3]
if len(ema):
    ax.plot(ema['ema_value'], ema['pk_ratio_low_k'], marker='o', label='low k')
    ax.plot(ema['ema_value'], ema['pk_ratio_mid_k'], marker='o', label='mid k')
    ax.plot(ema['ema_value'], ema['pk_ratio_high_k'], marker='o', label='high k')
if len(raw):
    ax.scatter([raw_x], [float(raw.iloc[0]['pk_ratio_low_k'])], marker='*', s=120, color='black')
ax.axhline(1.0, color='0.35', lw=1, ls='--')
ax.set_xscale('log')
ax.set_xlabel('EMA sigma_rel')
ax.set_ylabel('P(k) ratio')
ax.set_title('broad-band P(k) ratios, near 1 is better')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

fig.suptitle(f'{RUN_NAME}: raw vs post-hoc EMA ({SAMPLER})')
custom_plot = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_custom_metrics.png'
fig.savefig(custom_plot)
print('wrote', custom_plot)
plt.show()


## One-Point PDF Comparison

This overlays the real training reference and each generated sample set using the same bins. It catches cases where images look plausible but pixel distribution has shifted.


In [ ]:
real = load_real_lightweight(CONFIG_PATH, max_raw_samples=MAX_REAL_CUBES)
real_hist = field_histogram(evenly_limit(real, MAX_REAL_HIST), bins=160)
edges = np.asarray(real_hist['bin_edges'])
centers = 0.5 * (edges[:-1] + edges[1:])

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.step(centers, real_hist['hist'], where='mid', lw=2.2, color='black', label=f'real train ref n={min(len(real), MAX_REAL_HIST)}')
for rec in metrics.sort_values('ema_value', na_position='first').to_dict('records'):
    arr = evenly_limit(load_npz_samples(Path(rec['sample_path'])), int(rec['n_used']))
    hist = field_histogram(arr, bins=edges)
    label = f"{rec['ema_label']} n={int(rec['n_used'])}"
    ax.step(centers, hist['hist'], where='mid', lw=1.8, label=label)
ax.set_xlabel('normalized field value')
ax.set_ylabel('density')
ax.set_title(f'{RUN_NAME}: one-point PDF, raw vs EMA')
ax.grid(alpha=0.2)
ax.legend()
out = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_one_point_pdf.png'
fig.savefig(out)
print('wrote', out)
plt.show()


## P(k) Comparison

Left: mean P(k). Right: generated-to-real ratio. Ratios close to 1 across k are good. This is the main physics-style distribution check for whether EMA improved or hurt the generated fields.


In [ ]:
pk_real, k = batch_power_spectra(evenly_limit(real, MAX_REAL_PK), nbins=PK_NBINS)
real_mean = np.nanmean(pk_real, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(k, real_mean, color='black', lw=2.5, label='real train ref')
for rec in metrics.sort_values('ema_value', na_position='first').to_dict('records'):
    arr = evenly_limit(load_npz_samples(Path(rec['sample_path'])), int(rec['n_used']))
    pk_gen, _ = batch_power_spectra(arr, nbins=PK_NBINS)
    gen_mean = np.nanmean(pk_gen, axis=0)
    axes[0].plot(k, gen_mean, marker='o', ms=3, label=rec['ema_label'])
    axes[1].plot(k, gen_mean / np.clip(real_mean, 1e-30, None), marker='o', ms=3, label=rec['ema_label'])
axes[0].set_yscale('log')
axes[0].set_xlabel('k bin')
axes[0].set_ylabel('mean P(k)')
axes[0].set_title('mean power spectrum')
axes[1].axhline(1.0, color='0.35', lw=1)
axes[1].set_xlabel('k bin')
axes[1].set_ylabel('generated / real')
axes[1].set_title('P(k) ratio')
for ax in axes:
    ax.grid(alpha=0.25)
axes[1].legend()
out = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_pk_ratio.png'
fig.savefig(out)
print('wrote', out)
plt.show()


## Image Grid

These are not paired samples. To keep the grid readable after adding many EMA targets, this shows raw, the best one-point EMA, the best P(k) EMA, and a few reference targets if they exist.


In [ ]:
def first_channel(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 4:
        return arr[:, 0]
    if arr.ndim == 3:
        return arr
    raise ValueError(arr.shape)

candidate_labels = [
    'raw',
    str(best_hist['ema_label']),
    str(best_pk['ema_label']),
    'ema0p01',
    'ema0p02',
    'ema0p05',
    'ema0p10',
]
image_labels = []
for label in candidate_labels:
    if label in image_labels:
        continue
    path = sample_path(SAMPLE_ROOT, RUN_NAME, SEED, label, SAMPLER)
    if path.exists():
        image_labels.append(label)

if not image_labels:
    print('No sample files available for the image grid.')
else:
    n_show = 6
    fig, axes = plt.subplots(len(image_labels), n_show, figsize=(2.2 * n_show, 2.4 * len(image_labels)), constrained_layout=True)
    if len(image_labels) == 1:
        axes = np.asarray([axes])
    for r, label in enumerate(image_labels):
        path = sample_path(SAMPLE_ROOT, RUN_NAME, SEED, label, SAMPLER)
        arr = first_channel(load_npz_samples(path))[:n_show]
        for c in range(n_show):
            ax = axes[r, c]
            img = arr[c]
            lo, hi = np.percentile(img, [1, 99])
            ax.imshow(img, origin='lower', cmap='viridis', vmin=lo, vmax=hi)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f'{label} {c}')
    fig.suptitle(f'{RUN_NAME}: selected raw/EMA samples, per-field color scale')
    out = OUT_DIR / f'{RUN_NAME}_{SAMPLER}_ema_image_grid.png'
    fig.savefig(out)
    print('wrote', out)
    plt.show()


## Great Lakes Commands

To reconstruct the full dense EMA grid used by this notebook:

```bash
cd /home/jiamingp/diffusion_models_repo
RUN_NAME=nf_fig2_u64_d2p15_noaug_200k NUM_SAMPLES=128 OVERWRITE=0 sbatch -A huterer2 scripts/slurm/sample_nf_generalize_fig2_small_ema_one.sbatch
```

The default Slurm array is now `0-15%3` and covers:

`raw, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.008, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.075, 0.10`.

With `OVERWRITE=0`, existing files such as `raw`, `ema0p02`, and `ema0p10` are skipped and only missing EMA points are generated.
